# MEDAL: one prompt, MCTS initialization, then decoding

[Open in Colab](https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/MEDAL_MCTS_Demo.ipynb)

Run this notebook on its own. It loads **one LLaDA model**, shows the inline decomposition, inspects the **three initialization candidates**, and completes the selected partial state. The only reported metrics are NFE, wall-clock time, and remaining-mask entropy reduction.

This uses the repository's current MEDAL reconstruction. Decomposition is generated **inside the answer**, with no separate planning forward. The outline shown before search is the prompt template; the generated reasoning appears in the final response. No test set or answer scoring is run.

## Setup — run once

Use a fresh L4/A100 Colab runtime and the LLaDA dependency profile. This demo keeps weights in the notebook kernel so you can rerun decoding without reloading the model. Setup and checks are kept here; the action cells below stay short. The private-repository checkout uses the same Colab secret/hidden-token flow as the other notebooks.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local checkout: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
                str(REPO_DIR)], check=True)
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM") if IN_COLAB else REPO_DIR / ".cache/medal"
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
DEMONSTRATIONS_PATH = DRIVE_ROOT / "manifests/gsm8k_decomposition_train.json"

In [ ]:
import torch
from dlm_project.models.profiles import get_profile
from dlm_project.decoding.medal_demo import (
    DEFAULT_QUESTION, MCTSConfig, load_demo_model, prepare_prompt, show_prompt,
    run_initialization, show_candidates, show_metrics, run_completion,
)
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before loading LLaDA.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Use a GPU with bfloat16 support, such as L4 or A100.")
print("GPU:", torch.cuda.get_device_name(0))

### Model source and prompt settings

The source chooser first looks for the runtime copy, then the existing Drive snapshot, then uses the pinned Hub checkpoint. Set `MODEL_PATH` to another stored snapshot if needed. An existing Drive snapshot does not need a new download. Only the two training demonstrations are cached on first prompt preparation; existing demonstrations are reused offline.

Change `QUESTION`, or set `PROMPT_FILE` to a UTF-8 file containing your question. Defaults match the current MCTS implementation: 256 output positions, initialization depth 20, C=3, K1=3, K2=5, and no search cap.

In [ ]:
model_name = get_profile("llada")["repo_id"].split("/")[-1]
stored_models = [Path("/content/models") / model_name, DRIVE_ROOT / "models" / model_name]
MODEL_PATH = next((str(path) for path in stored_models if (path / "config.json").is_file()),
                  get_profile("llada")["repo_id"])
QUESTION = DEFAULT_QUESTION
PROMPT_FILE = None
GEN_LENGTH, SEED, TEMPERATURE = 256, 1, 1.0
MCTS_SETTINGS = MCTSConfig(init_tokens=20, num_candidates=3, top_k_tokens=3, top_k_actions=5)
print("Model source:", MODEL_PATH)

## 1. Load the model

Run once. Initialization and every later completion share this model.

In [ ]:
llada = load_demo_model(model_path=MODEL_PATH, device="cuda",
                        local_files_only=Path(MODEL_PATH).is_dir())
model, tokenizer = llada.model, llada.tokenizer

## 2. Load the prompt and show its decomposition

The full two-shot prompt is available as `prompt`; `decomposition` contains its metadata. The display below shows your question and the three-subtask guidance. There is no separately generated plan.

In [ ]:
question = Path(PROMPT_FILE).read_text(encoding="utf-8") if PROMPT_FILE else QUESTION
prompt, decomposition = prepare_prompt(question, DEMONSTRATIONS_PATH,
                                       cache_dir=os.environ["HF_DATASETS_CACHE"])
prompt_ids = llada.encode_prompt(prompt)
show_prompt(question)

## 3. Initialize a partial state with the current MCTS

This cell performs search only. Keep `initialization` to reuse its selected partial state when changing the later sampler.

In [ ]:
initialization = run_initialization(
    model, prompt_ids, llada.mask_id, gen_length=GEN_LENGTH,
    mcts=MCTS_SETTINGS, seed=SEED, tokenizer=tokenizer,
)

### 3.1. Top three C candidates

Each candidate lists only its unmasked output positions and token values, with adjusted token confidence `s`. `Q` is the cached selection value used by the current MCTS: the **last parent-to-child normalized entropy reduction**. The winning candidate is marked. The root-to-candidate entropy change is shown separately for every candidate; it does not change selection. These displays use cached statistics and add no NFE.

In [ ]:
show_candidates(initialization, tokenizer=tokenizer)

### 3.2. Initialization cost

NFE counts actual model forwards. Selection, expansion, backpropagation, and cached candidate ranking add zero forwards. Wall-clock timing synchronizes CUDA and excludes model loading, prompt preparation, and these display cells.

In [ ]:
show_metrics(initialization)

## 4. Complete the selected state

`"medal"` uses the current adjusted-score softmax completion. Copy this cell and change only `DECODING_METHOD` when adding a sampler. Every call receives a fresh copy of the same selected partial state; it preserves the prompt and MCTS commitments. Rerunning this cell performs completion only.

In [ ]:
DECODING_METHOD = "medal"  # Later: "my_sampler:decode" or an imported callable.
completion = run_completion(initialization, method=DECODING_METHOD, seed=SEED,
                            temperature=TEMPERATURE, tokenizer=tokenizer)
print(tokenizer.decode(completion.token_ids[0, prompt_ids.shape[1]:].tolist(), skip_special_tokens=True))

In [ ]:
show_metrics(initialization, completion)

### Adding your sampler later

Put `my_sampler.py` in `REPO_DIR` (or another importable directory) and set `DECODING_METHOD = "my_sampler:decode"`. You can also pass an imported function directly. Its interface is:

```python
def decode(model, state, prompt_len, mask_id, config, *, seed, temperature, tokenizer):
    ...
```

`state` is a CPU `torch.long` tensor shaped `[1, prompt_length + output_length]`. Move inference inputs to `model.device` as needed, fill its remaining output masks, and return an object with `token_ids` of the same shape (`CompletionResult` is already available in `dlm_project.decoding.mcts_decoding`). A `trace` list is optional. Use the supplied model so every forward is counted. The demo verifies that the prompt and initialized tokens survive; checks stay inside the helper.

Each completion report includes the initialization cost **once** plus that completion's actual forwards and time. Earlier sampler trials are not added. The final remaining-mask entropy is zero because no masks remain; it is not a correctness score.

Python entry point for the same flow:

```bash
python scripts/demo_medal.py --model-path /path/to/LLaDA-8B-Instruct --local-files-only \
    --prompt-file /path/to/question.txt
```